![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 03 | Evaluate — the Report

*Project 3 — AI Product: From Model to Product*

**Team:** Zoé Doucet

**Track:** LLM

**Data:** Forest Entomology and Pathology

---

**This notebook is your report.** It is read on its own, by someone who has not seen notebooks 01 and 02: the question, the evidence, what you concluded and how far to trust it.

> **Done when:** the final version is evaluated once, its failures are analysed with examples, its cost is known, and the conclusions answer the question you framed in notebook 01.

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")  # Allow imports from src/

import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from openai import OpenAI

from src.functions import *

# Notebook display settings
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

In [2]:
# Loading the evaluation questions
eval_df = pd.read_csv("../data/evaluation.csv")

In [ ]:
# Loading the final configuration from Notebook 02
config_path = Path("../models/rag_config.json")

with open(config_path, "r", encoding="utf-8") as file:
    config = json.load(file)

# Loading the selected embedding model
model = SentenceTransformer(config["embedding_model"])

# Reopening the persistent Chroma database
chroma_client = chromadb.PersistentClient(path=config["chroma_path"])

# Opening the existing collection
collection = chroma_client.get_collection(name=config["collection_name"])

# Verifying the loaded configuration
print("Embedding model:", config["embedding_model"])
print("Embedding dimensions:", model.get_embedding_dimension())
print("Stored chunks:", collection.count())
print("Retrieval k:", config["k"])
print("LLM:", config["llm_model"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model: sentence-transformers/all-mpnet-base-v2
Embedding dimensions: 768
Stored chunks: 3274
Retrieval k: 5
LLM: gpt-6-luna


In [5]:
# Initializeing the OpenAI client
load_dotenv(Path("..") / ".env")

openai_client = OpenAI()

---
## 1. The question

### Project overview

The **Forest Pest Assistant** is a textbook-grounded chatbot designed for non-experts, such as garden owners and people managing small forest plots. Users can ask questions about tree health or describe symptoms of possible insect infestations. The assistant provides evidence-based explanations, asks clarifying questions when necessary, and helps users decide whether further investigation or professional advice is needed.

The chatbot uses *Forest Entomology and Pathology, Volume 1: Entomology* as its sole knowledge source. It is designed to avoid unsupported diagnoses, pesticide recommendations, and predictions about tree survival. When the available evidence is insufficient, it should communicate uncertainty rather than provide potentially misleading advice.

### Baseline and final model

The initial **TF-IDF baseline** achieved a page-level retrieval hit rate of **25.0%** on 12 answerable evaluation questions.

Two semantic embedding models, MiniLM and MPNet, were subsequently compared using `k=3` and `k=5`. The selected configuration uses:

- **Embedding model:** `sentence-transformers/all-mpnet-base-v2`
- **Vector database:** Chroma
- **Retrieval setting:** `k=5`
- **Text chunks:** 150 words with a 30-word overlap
- **Language model:** `gpt-6-luna`

The selected MPNet configuration achieved the highest page-level retrieval hit rate of **50.0% (6/12 questions)**. This improves on the TF-IDF baseline, although retrieving the correct page does not always guarantee that the exact supporting passage reaches the language model.

### Evaluation objective

The final assistant is evaluated using **15 predefined questions**, including 12 answerable and 3 unanswerable cases. The evaluation examines factual correctness, relevance, citation support, appropriate abstention, response time, and cost.

The main priority is to minimize unsupported or misleading answers. A cautious refusal is preferable to an incorrect diagnosis or recommendation when the textbook does not provide sufficient evidence.


---
## 2. The final evaluation

- **LLM track:** the whole evaluation set. Report separately: the retrieval hit rate (or the share of questions with the right tools), whether answers are correct and cite the right source, and whether the out-of-scope questions are refused.

---
## 3. Where it goes wrong

- **LLM track:** every failure classified as a retrieval (or tool) failure or a generation failure, with examples.

Then: where should your product **not** be trusted?

---
## 4. Cost and speed

- **LLM track:** tokens, cost and time per question, and the total for the evaluation. What would 1,000 questions a day cost per month?

---
## 5. The saved model or pipeline

Reload what you saved in notebook 02 and use it on a new example, exactly as the app does.

---
## 6. Conclusions and limitations

Would you ship it, to whom, and with what warning? What data, products or questions does it hold for? What would you do next?